In [1]:
import os

# Fix for protobuf incompatibility with keras 3.x
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import pandas as pd
import numpy as np
import imageio
from keras import Model, Input
from keras.layers import (
    Conv2D,
    BatchNormalization,
    ReLU,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Activation,
)



2026-05-07 16:24:48.165879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778171088.180082      56 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778171088.184258      56 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 16:24:48.200131: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
BASE_DIR = os.path.abspath("../input/aerial-cactus-identification")
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))




In [3]:
def load_images(df, folder):
    """
    Load 32x32 RGB images from `folder` using filenames from `df.id`.
    Skips any entries that are not image files.
    Grayscale images are converted to 3‑channel RGB.
    Images are normalized to have mean≈0 and std≈1.
    """
    # keep only .jpg/.jpeg/.png files
    img_files = [
        f
        for f in df["id"]
        if isinstance(f, str) and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    n = len(img_files)
    images = np.zeros((n, 32, 32, 3), dtype=np.float32)
    for i, filename in enumerate(img_files):
        path = os.path.join(folder, filename)
        img = imageio.imread(path).astype(np.float32)
        # If image is grayscale (H, W) convert to (H, W, 3)
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        # If image has alpha channel, drop it
        if img.shape[-1] == 4:
            img = img[..., :3]
        images[i] = img
    # Normalization (centered around 0, scale ~1)
    return (images - 128.0) / 64.0


images = load_images(df, TRAIN_DIR)




/tmp/ipykernel_56/1322939607.py:18: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path).astype(np.float32)


In [4]:
def ConvCell(m, filters, kernel=3):
    m = Conv2D(filters, kernel, padding="same")(m)
    m = BatchNormalization()(m)
    m = ReLU()(m)
    return m


def DeepConvCell(m, n, filters, kernel=3):
    for _ in range(n):
        m = ConvCell(m, filters, kernel)
    m = MaxPooling2D()(m)
    return m


n_inp = Input(shape=(32, 32, 3))
conv0 = DeepConvCell(n_inp, 3, 32)
conv1 = DeepConvCell(conv0, 2, 64)
conv2 = DeepConvCell(conv1, 1, 128)

convF = Conv2D(1, 3, padding="same")(conv2)
gl_avg_pool = GlobalAveragePooling2D()(convF)
fc = Activation("sigmoid")(gl_avg_pool)

m = Model(inputs=n_inp, outputs=fc)
m.compile(loss="binary_crossentropy", optimizer="adam")
# m.summary()  # optional, can be commented out to reduce console output



2026-05-07 16:25:14.222482: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778171114.222931      56 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:86:00.0, compute capability: 8.6


In [5]:
X_train = images.astype(np.float32)
y_train = df["has_cactus"].values.astype(np.float32)



In [6]:
m.fit(
    X_train,
    y_train,
    batch_size=64,
    epochs=50,
    validation_split=0.33,
    verbose=2,
)



Epoch 1/50


I0000 00:00:1778171118.591860     112 service.cc:148] XLA service 0x7f1768018590 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778171118.591892     112 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-07 16:25:18.645396: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778171118.939458     112 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778171120.946937     112 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


149/149 - 9s - 63ms/step - loss: 0.1220 - val_loss: 0.6218
Epoch 2/50
149/149 - 1s - 4ms/step - loss: 0.0608 - val_loss: 0.1696
Epoch 3/50
149/149 - 1s - 4ms/step - loss: 0.0426 - val_loss: 0.0247
Epoch 4/50
149/149 - 1s - 4ms/step - loss: 0.0363 - val_loss: 0.0264
Epoch 5/50
149/149 - 1s - 4ms/step - loss: 0.0287 - val_loss: 0.1417
Epoch 6/50
149/149 - 1s - 4ms/step - loss: 0.0248 - val_loss: 0.0234
Epoch 7/50
149/149 - 1s - 4ms/step - loss: 0.0163 - val_loss: 0.0265
Epoch 8/50
149/149 - 1s - 4ms/step - loss: 0.0100 - val_loss: 0.0182
Epoch 9/50
149/149 - 1s - 4ms/step - loss: 0.0130 - val_loss: 0.0283
Epoch 10/50
149/149 - 1s - 4ms/step - loss: 0.0055 - val_loss: 0.0330
Epoch 11/50
149/149 - 1s - 4ms/step - loss: 0.0048 - val_loss: 0.0229
Epoch 12/50
149/149 - 1s - 4ms/step - loss: 0.0051 - val_loss: 0.0102
Epoch 13/50
149/149 - 1s - 4ms/step - loss: 0.0035 - val_loss: 0.0110
Epoch 14/50
149/149 - 1s - 4ms/step - loss: 0.0094 - val_loss: 0.0099
Epoch 15/50
149/149 - 1s - 4ms/step - l

In [7]:
# Prepare test dataframe – ensure only image files are used
test_ids = sorted(
    [f for f in os.listdir(TEST_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
)
df_test = pd.DataFrame({"id": test_ids})

test_images = load_images(df_test, TEST_DIR)

preds = m.predict(test_images, batch_size=64).flatten()

df_test["has_cactus"] = preds.astype(np.float32)



/tmp/ipykernel_56/1322939607.py:18: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path).astype(np.float32)


52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


In [8]:
# Write submission with the correct column name
submission_path = "submission.csv"
df_test.to_csv(submission_path, index=False)